# Laboratorio 03 — El folio que llegó dos veces

**Duración: 60 minutos.**

Hasta hoy escribiste tres o cuatro filas a mano. **Este laboratorio trabaja con
documentos tributarios de verdad**, del padrón de la DGT, y con el problema que su
área de datos tiene todos los días.

---

### El problema

El flujo de recepción de documentos trae **reenvíos**. Un contribuyente emite una
factura, no recibe el acuse, y la manda otra vez. El mismo folio llega dos veces.

Eso, en la tabla de la DGT, se convierte en dos filas para un solo documento. Y dos
filas para un solo documento significan que el total del mes está inflado.

Las dos salidas que la DGT usa hoy son malas:

- **Insertar sin más** y arreglarlo después: las cifras salen mal mientras tanto.
- **Borrar todo el periodo y recargarlo**: carísimo, y la tabla pierde su historia.

Lo que hace falta es otra cosa: una operación que mire cada documento que llega y
**decida por sí sola** qué hacer con él. Si ya está, corregirlo. Si no está,
agregarlo.

### En la libreta

El proveedor entrega dos veces la misma factura. El almacenero **no la anota dos
veces**, y tampoco rehace la libreta entera: busca la anotación que ya existe y la
corrige; y si no la encuentra, la agrega.

Esa decisión —una por documento— es todo lo que hace la operación de fusión.

---

### Cómo se usa este notebook

Como los anteriores: `%%sql` viene en la primera línea, tú escribes la sentencia
debajo, una por celda.

**De dónde salen los datos.** Los documentos ya están cargados en el ambiente, en un
espacio compartido llamado `curso`. Tú **lees** de ahí y **escribes** en tu propia
database, así que nadie le toca el dato a nadie.

**Recuerda el nombre a tres partes** para las vistas del sistema:
`mi_espacio.documentos_lab03.snapshots`, aunque ya hayas hecho `USE`.


## Paso 0 — Preparar la tabla

Este laboratorio no depende de ninguno anterior. Primero, a tu espacio de
trabajo.

In [1]:
%%sql
-- Celda 0.1
USE mi_espacio

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/24 21:13:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Partimos de cero, por si ya lo habías corrido.

In [2]:
%%sql
-- Celda 0.2
DROP TABLE IF EXISTS documentos_lab03

Ahora la tabla del laboratorio, y aquí hay algo nuevo: en vez de declarar las
columnas una por una, la creamos **a partir de una consulta**.

`CREATE TABLE ... USING iceberg AS SELECT ...` crea la tabla con las columnas y los
tipos que devuelva ese `SELECT`, y de paso la carga. Es como se cargan los datos de
verdad: nadie teclea un padrón a mano.

`curso.recepcion_lote1` es el primer lote de documentos recibidos, ya cargado en el
ambiente.

In [3]:
%%sql
-- Celda 0.3
CREATE TABLE documentos_lab03 USING iceberg AS
SELECT * FROM curso.recepcion_lote1

¿Cuántos documentos quedaron?

In [4]:
%%sql
-- Celda 0.4
SELECT count(*) AS documentos FROM documentos_lab03

documentos
30


Treinta documentos. Miremos unos pocos para saber con qué estamos trabajando.

Cada fila es un documento tributario electrónico: quién lo emitió, de qué tipo es
(33 factura afecta, 34 exenta, 39 boleta, 52 guía de despacho, 56 nota de débito,
61 nota de crédito), su folio, cuándo se emitió, **cuándo lo recibió la DGT**, el
monto y en qué estado quedó.

In [5]:
%%sql
-- Celda 0.5
SELECT rut_emisor, tipo_dte, folio, fecha_recepcion, monto_total, estado_sii
FROM documentos_lab03
ORDER BY rut_emisor, tipo_dte, folio
LIMIT 5

rut_emisor,tipo_dte,folio,fecha_recepcion,monto_total,estado_sii
76030854-4,39,864,2026-06-06 17:23:16,29857.10,ACEPTADO
76034637-3,39,860,2026-06-08 03:17:22,70864.50,ACEPTADO
76057484-8,33,226,2026-06-12 00:29:09,6972860.93,ACEPTADO
76133016-0,61,12,2026-06-10 03:23:20,-895572.58,ACEPTADO
76252637-9,33,972,2026-06-07 08:21:09,17190434.17,ACEPTADO


## Paso 1 — Llega el lote nuevo

Al día siguiente llega el segundo lote de recepción. Está en
`curso.recepcion_lote2`. Veamos cuántos documentos trae.

In [6]:
%%sql
-- Celda 1.1
SELECT count(*) AS documentos FROM curso.recepcion_lote2

documentos
20


Veinte. Pero la pregunta importante es otra: **¿cuántos de esos veinte ya los
tenemos?**

Un documento tributario se identifica por tres cosas juntas: **quién lo emitió, de
qué tipo es, y su número de folio**. Esas tres columnas son su *llave natural*: dos
filas con los mismos tres valores son el mismo documento.

Crucemos los dos lotes por esa llave.

In [7]:
%%sql
-- Celda 1.2
SELECT count(*) AS ya_los_teniamos
FROM curso.recepcion_lote2 AS nuevo
JOIN documentos_lab03 AS actual
  ON  actual.rut_emisor = nuevo.rut_emisor
  AND actual.tipo_dte   = nuevo.tipo_dte
  AND actual.folio      = nuevo.folio

ya_los_teniamos
12


## Paso 2 — El error que hay que ver

**Doce de los veinte ya están en la tabla.** Son los reenvíos.

Ahora vamos a hacer, a propósito, lo que no hay que hacer: insertar el lote nuevo tal
cual. No es un truco; es exactamente la sentencia que uno escribiría sin pensarlo, y
conviene ver qué produce **en tu propia tabla** antes de aprender la alternativa.

In [8]:
%%sql
-- Celda 2.1
INSERT INTO documentos_lab03 SELECT * FROM curso.recepcion_lote2

Contemos.

In [9]:
%%sql
-- Celda 2.2
SELECT count(*) AS documentos FROM documentos_lab03

documentos
50


Cincuenta: treinta más veinte. Aritmética impecable y resultado equivocado, porque
doce de esos documentos son el mismo documento contado dos veces.

Busquémoslos. Agrupamos por la llave natural y nos quedamos con los grupos que
tienen más de una fila.

In [10]:
%%sql
-- Celda 2.3
SELECT rut_emisor, tipo_dte, folio, count(*) AS veces
FROM documentos_lab03
GROUP BY rut_emisor, tipo_dte, folio
HAVING count(*) > 1
ORDER BY rut_emisor, tipo_dte, folio

rut_emisor,tipo_dte,folio,veces
76057484-8,33,226,2
76133016-0,61,12,2
76705019-4,34,149,2
76899472-2,61,28,2
76913181-7,39,778,2
77612645-4,61,100,2
77781727-2,39,15,2
78086012-K,39,223,2
78914770-1,61,10,2
79068554-7,33,52,2


Doce folios repetidos. Miremos uno de cerca —el primero de la lista— para ver en
qué se diferencian las dos filas.

El `ORDER BY` del final no sobra: sin él, las dos filas pueden salir en cualquier
orden y no sería igual en todas las pantallas de la sala.

In [11]:
%%sql
-- Celda 2.4
SELECT rut_emisor, tipo_dte, folio, fecha_recepcion, monto_total, estado_sii
FROM documentos_lab03
WHERE rut_emisor = '76057484-8' AND tipo_dte = 33 AND folio = 226
ORDER BY fecha_recepcion

rut_emisor,tipo_dte,folio,fecha_recepcion,monto_total,estado_sii
76057484-8,33,226,2026-06-12 00:29:09,6972860.93,ACEPTADO
76057484-8,33,226,2026-06-13 15:41:09,6972860.93,ACEPTADO


## Paso 3 — Deshacer

Ahí está el problema, con nombre y apellido: **el mismo documento, dos veces**, con
la única diferencia de cuándo lo recibió la DGT. El monto está contado dos veces.

Antes de aprender a hacerlo bien, deshagamos esto. Y no vamos a borrar filas a mano:
ya sabes que la tabla guarda sus páginas.

Pidamos la lista.

In [12]:
%%sql
-- Celda 3.1
SELECT snapshot_id, committed_at, operation
FROM mi_espacio.documentos_lab03.snapshots
ORDER BY committed_at

snapshot_id,committed_at,operation
92422959594958504,2026-09-24 21:13:52.926,append
7112055896236960535,2026-09-24 21:13:54.844,append


Dos páginas: la carga inicial y el `INSERT` que acabas de hacer.

**Copia el `snapshot_id` de la primera** —la de arriba— y vuelve a ella. Es
exactamente lo que hiciste en el laboratorio 01, ahora sobre datos de verdad y por un
motivo real: un proceso cargó mal y hay que deshacerlo.

In [13]:
%%sql
-- Celda 3.2
CALL spark_catalog.system.rollback_to_snapshot('mi_espacio.documentos_lab03', <TU_SNAPSHOT_ID>)

previous_snapshot_id,current_snapshot_id
7112055896236960535,92422959594958504


Comprobemos.

In [14]:
%%sql
-- Celda 3.3
SELECT count(*) AS documentos FROM documentos_lab03

documentos
30


## Paso 4 — La fusión

Treinta otra vez. La carga mala quedó deshecha en una sentencia, sin restaurar
nada de un respaldo.

Ahora sí, la forma correcta. Léela por partes **antes** de escribirla:

| Parte | Qué dice |
|---|---|
| `MERGE INTO documentos_lab03 AS destino` | La tabla que se va a modificar |
| `USING curso.recepcion_lote2 AS origen` | De dónde vienen las filas que llegan |
| `ON destino.rut_emisor = origen.rut_emisor AND ...` | **Cuándo dos filas son el mismo documento**: la llave natural, las tres columnas juntas |
| `WHEN MATCHED THEN UPDATE SET ...` | Qué hacer si el documento **ya estaba**: corregir la fecha de recepción con la del reenvío |
| `WHEN NOT MATCHED THEN INSERT *` | Qué hacer si **no estaba**: agregarlo entero |

Eso es todo. Una decisión por documento, y la toma la base de datos, no tú.

In [15]:
%%sql
-- Celda 4.1
MERGE INTO documentos_lab03 AS destino
USING curso.recepcion_lote2 AS origen
  ON  destino.rut_emisor = origen.rut_emisor
  AND destino.tipo_dte   = origen.tipo_dte
  AND destino.folio      = origen.folio
WHEN MATCHED THEN UPDATE SET destino.fecha_recepcion = origen.fecha_recepcion
WHEN NOT MATCHED THEN INSERT *

Contemos otra vez.

In [16]:
%%sql
-- Celda 4.2
SELECT count(*) AS documentos FROM documentos_lab03

documentos
38


**Treinta y ocho**, no cincuenta. Los doce reenvíos no agregaron filas: corrigieron
las que ya estaban. Los ocho documentos nuevos sí entraron. 30 + 8 = 38.

Volvamos al folio de antes.

In [17]:
%%sql
-- Celda 4.3
SELECT rut_emisor, tipo_dte, folio, fecha_recepcion, monto_total, estado_sii
FROM documentos_lab03
WHERE rut_emisor = '76057484-8' AND tipo_dte = 33 AND folio = 226

rut_emisor,tipo_dte,folio,fecha_recepcion,monto_total,estado_sii
76057484-8,33,226,2026-06-13 15:41:09,6972860.93,ACEPTADO


**Una sola fila**, y con la fecha de recepción del reenvío —la más tardía de las dos
que viste—: el dato quedó actualizado, no duplicado.

Confirmemos que ya no queda ningún repetido en toda la tabla.

In [18]:
%%sql
-- Celda 4.4
SELECT count(*) AS folios_repetidos FROM (
    SELECT rut_emisor, tipo_dte, folio
    FROM documentos_lab03
    GROUP BY rut_emisor, tipo_dte, folio
    HAVING count(*) > 1
)

folios_repetidos
0


## Paso 5 — Las otras dos operaciones

Cero. La tabla quedó correcta.

Para cerrar el conjunto de operaciones que vas a usar en tu trabajo, faltan dos que
ya conoces de cualquier base de datos, y que en Iceberg funcionan igual.

La primera: **corregir filas que cumplen una condición**. Marquemos como revisadas
todas las notas de crédito, que son los documentos de tipo 61.

In [19]:
%%sql
-- Celda 5.1
UPDATE documentos_lab03 SET estado_sii = 'REVISADO' WHERE tipo_dte = 61

Veamos a cuántas afectó.

In [20]:
%%sql
-- Celda 5.2
SELECT estado_sii, count(*) AS documentos
FROM documentos_lab03
GROUP BY estado_sii
ORDER BY estado_sii

estado_sii,documentos
ACEPTADO,31
ACEPTADO_CON_REPAROS,2
REVISADO,5


Cinco documentos quedaron en `REVISADO`.

Y la segunda: **borrar filas que cumplen una condición**. Saquemos los documentos de
monto negativo, que en este lote son las notas de crédito.

In [21]:
%%sql
-- Celda 5.3
DELETE FROM documentos_lab03 WHERE monto_total < 0

Y el recuento final.

In [22]:
%%sql
-- Celda 5.4
SELECT count(*) AS documentos FROM documentos_lab03

documentos
33


---

## Listo

Treinta y tres. Cinco documentos menos, los que borraste.

### Lo que te llevas

| Lo que necesitas | La sentencia |
|---|---|
| Cargar una tabla desde una consulta | `CREATE TABLE … USING iceberg AS SELECT …` |
| Agregar filas | `INSERT INTO … SELECT …` |
| **Cargar sin duplicar** | `MERGE INTO … USING … ON … WHEN MATCHED … WHEN NOT MATCHED …` |
| Corregir filas | `UPDATE … SET … WHERE …` |
| Borrar filas | `DELETE FROM … WHERE …` |
| Deshacer una carga entera | `CALL … rollback_to_snapshot(…)` |

De todas, la que cambia el trabajo del área de datos de la DGT es la fusión. Antes de
Iceberg, cargar un lote sin duplicar significaba borrar la partición del periodo y
reconstruirla completa: horas de proceso y la historia perdida. Ahora es una
sentencia que decide documento por documento.

Y fíjate en lo que hiciste en el paso 3: **deshiciste una carga mala volviendo a una
página anterior**. Eso no lo aprendiste hoy, lo aprendiste en el laboratorio 01. Así
se va a ir armando el curso: lo que ya sabes se usa, no se vuelve a explicar.

### La pregunta del laboratorio siguiente

Tu tabla tiene 33 documentos y todo responde al instante. La de la DGT tiene
**dos millones**, y crece todos los días.

Cuando Valentina pregunta por el total de un solo día de junio, ¿qué tiene que hacer
la tabla para responderle? Con lo que viste en el laboratorio 02 ya lo puedes
sospechar: la tabla es un montón de archivos, y **para encontrar un día habría que
abrirlos todos**.

**¿Cómo se le dice a la tabla dónde buscar?** Eso es el **Laboratorio 04 — Buscar sin
dar vuelta el almacén**.
